# Milestone 6 Lab, notebook 8 of 8: The whole system

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| **8. The whole system** (you are here) | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **about 10**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

This is the last notebook of the project.

In milestone 1 an AI analyst answered one question from a table of prepared numbers. Everything behind those numbers was pretend. Since then you have replaced the pretend parts one at a time.

Here you run the whole thing once, from files to answer, and look at what stands behind that answer now.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file.

In [ ]:
import time

from lab_helpers import LAB_E2E, MAIN_QUESTION, health_table, show_answer, show_doc

## Step 1: From nothing to month-end close

One call. Count what it does.

In [ ]:
from retail_profit_stack.e2e import build_close

start = time.time()
workspace = build_close(LAB_E2E)

print("Built in", round(time.time() - start, 1), "seconds, with no model")

In [ ]:
from retail_profit_stack.ingest import ledger
from retail_profit_stack.lineage.runs import read_history

history = read_history(workspace.warehouse)
close = history[-1]

print("Milestone 5: files loaded       ", ledger.outcome_counts(workspace.places.state_file))
print("Milestone 3: checks             ", close["check_outcomes"])
print("Milestone 3: rows set aside     ", sum(close["quarantined"].values()))
print("Milestone 2: tables calculated  ", 6)
print("Milestone 4: lineage adds up    ", close["lineage_adds_up"])
print("Milestone 4: runs kept          ", [record["run_id"] for record in history])
print("Milestone 6: this run's health  ", close["health"]["warnings"])

## Step 2: The leader's question, at close

The same question as milestone 1, word for word.

In [ ]:
from retail_profit_stack.agent import run_analyst
from retail_profit_stack.store import Store


def store_for(run_id=None):
    return Store(source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                 run_id=run_id, state_file=workspace.places.state_file)


print(MAIN_QUESTION)

at_close = run_analyst(MAIN_QUESTION, store=store_for())
show_answer(at_close.answer)

In [ ]:
print("Verified by code:", at_close.verification.passed, "  Run:", at_close.answer.run_id)
print("Tools called:", [call.name for call in at_close.tool_log])
print("Cost:", at_close.cost, "US dollars")

If a check failed, run the cell that asks again.

Confidence is **low**, and the answer says why: the camp stove has no cost. That warning comes from a real check on real records, loaded from real files.

## Step 3: Life goes on

A data engineer decides a finding at the review desk. Three files arrive late. The pipeline runs again, incrementally.

In [ ]:
from lab_helpers import LAB_DECISIONS

from retail_profit_stack.make_history import apply_late_files
from retail_profit_stack.quality.review import decide

if LAB_DECISIONS.exists():
    LAB_DECISIONS.unlink()
decide(LAB_DECISIONS, "refund_exceeds_line|refunds|REF-0002", "exclude", "your name", "Typed wrongly")

late_run = apply_late_files(workspace.places, workspace.warehouse, workspace.late, decisions_file=LAB_DECISIONS)

print(late_run.run_id, "-", late_run.record["mode"], "- months:", late_run.record["months_recalculated"])
health_table(late_run.record)

The mode says **full**, not incremental. A decision was made at the desk, and a decision can move a record in or out of the figures, so the pipeline rechecked everything. It chose the safe path by itself.

## Step 4: The same question, after

Ask again. The words are the same.

In [ ]:
after = run_analyst(MAIN_QUESTION, store=store_for())

show_answer(after.answer)

In [ ]:
print("At close: confidence", at_close.answer.confidence.level, " run", at_close.answer.run_id)
print("Now:      confidence", after.answer.confidence.level, " run", after.answer.run_id)
print("Verified by code:", after.verification.passed)

The confidence changed, because the data behind it changed: the cost arrived, and the oversized refund was dealt with. Nothing was edited by hand. The answer followed the records.

And the leader can ask what happened.

In [ ]:
why = run_analyst("Which products' contribution profit changed between the last two pipeline runs?", store=store_for())

show_answer(why.answer)

## Step 5: What stands behind one number

Take one figure from the answer and walk it all the way back.

In [ ]:
from retail_profit_stack import tools

store = store_for()
row = tools.query_profit("2026-10", "product", top_n=1, store=store).rows[0]
evidence = tools.get_evidence("2026-10", row.entity_id, store=store)

print("The figure:   ", row.entity_name, row.contribution_profit)
print("Its run:      ", evidence.run_id)
print("Its records:  ", evidence.total_records, evidence.record_counts)
print("Set aside:    ", evidence.set_aside_count, "duplicates, kept and not counted")
print("Its steps:    ", evidence.steps[-1])
print("One record:   ", evidence.records[0].description)

In milestone 1 that figure was a number typed into a table. Now it is the sum of several hundred records, each from a named file, checked, and provably added up.

## Step 6: Six milestones

| | Milestone 1 | Now |
|---|---|---|
| The figures | Typed into a table | Calculated from records, on two engines |
| The warnings | A hand-written list | Five checks, with a fixed outcome each |
| A warning's effect | Something to read | Bad records set aside; a person decides the doubtful ones |
| The evidence | A few sample rows | Every record, traced, and proved to add up |
| "Why did it change?" | No answer | Looked up in the run history, down to the file |
| How records arrive | They were simply there | Files, loaded repeatably, late and corrected ones too |
| Measuring the analyst | Seven questions | 28 questions, two sets, a verifier and a judge |
| Improving the analyst | A person reads failures | An agent proposes, a person approves, code decides |

One thing never changed, from the first notebook to this one: **the model never produced a number.** Code calculated every figure, decided every outcome, and checked every answer. AI drafted, suggested, investigated, explained, judged, and proposed. A person approved what mattered.

## Step 7: From the lab to production

In [ ]:
show_doc("LLD.md", "12. Where each part is built")

To build everything and run the tests, open a terminal in this milestone's folder and run:

```
uv run python -m retail_profit_stack.make_history --late
uv run pytest
```

To see the whole pipeline working on files you choose, from the project root:

```
uv run python pipeline_live/server.py
```

## Your assignment

1. **A second lever.** Add a third model to `ALLOWED_MODELS` in `retail_profit_stack/improver.py`. Change the first recipe so that when there are no mistakes, the improver tries the cheaper model. Run a round with the full question sets. Is the cheaper model kept? Which score decided it?
2. **A harder desk.** At the review desk, a decision applies to one record. Add a decision that applies to a whole check for one product, such as "accept every out-of-order shipment for the rain jacket". What should happen when a new record of that kind arrives next month?
3. **Break a layer.** In a copy of the landing folder, delete one orders file, then run notebook 6's evaluation on it. Which questions fail, and how far up the stack does the damage show? This is what an end-to-end evaluation is for.
4. **Your own business.** Pick a question a leader in your own work asks every month. Which of the six layers already exist for it, and which are still pretend?

## What we now know

- The whole stack builds from nothing in about a second, and every layer is real.
- The same question gave two answers at two moments, each naming its run, each true.
- Every number in an answer can be walked back to named records in named files.
- Code decides and checks. AI drafts, explains, and proposes. A person approves.

You have finished Retail Profit Stack. Open `milestone_6_recap.html` in a browser.